# Blindspots — proof of concept

A person says one thing. We show them the tetrad around it, with the corner they are
not looking at — **A+**, the constructive side of the opposing view — in front.

This notebook is the pre-MVP: no app, no server, no UI. It exists to answer two
questions with real output before anyone builds a backend:

1. Is the A+ the framework produces worth showing to a person?
2. Which of the two ways of producing it should the app run?
3. Can a person POLISH their way to the one that lands — react to a card, get a
   different or adjusted one, without it becoming a chat?

**Where it stands (2026-10-02, Sonnet 5, 40 utterances, the pipeline's own coherence
judge and auditors — `docs/dev-notes/antithesis-selection.md`, "the three doors" and
"The one-shot build"):** Path A draws a coherent first tetrad for 25/40 at ~7 s with a
genuine opposing position for 39/40. Path B — the Advisor's `anchor` — used to run a
staged pipeline (headline → antithesis ladder → aspects for a pair never seen whole) that
drew 14/40 coherent at ~55 s with mostly strawman antitheses; since `fac4909` it reasons
the tetrad in ONE call, the same call Path A makes, and then classifies, scores, grounds,
validates and persists it: 21/40 and 21/40 coherent on two generations, 36–38/40 genuine
antitheses, ~43 s. The two paths now draw the same tetrad; Path B adds the graph.
**Since 2026-10-04 (`e3cc495`) the agent checks its own tetrad before the person sees it:**
both paths draw THREE tetrads in parallel and the framework's own coherence check keeps the
one that holds best (`concerns/tetrad_candidates.py`; selection only, never regeneration from
the verdict). Pre-registered, same 40 utterances, fresh judge: Path A 36/40 coherent (from
25/40; 12 improved, 0 worsened) at ~15 s, genuine antitheses 36/40; Path B 30/40 (from 20/40;
13 improved, 3 worsened) at ~34 s, genuine antitheses 36/40. The two runners-up of a build
come back with their verdicts (`SketchTetrad.alternatives`) and can be persisted without a
new reasoning call (`SketchTetrad(sketch=)`) — Path B's polishing uses them below.
**Since 2026-10-04 (later the same day) the entry is GATED:** on 20 questions
(`tests/e2e/probe_question_entry.py`) the view turn drew 20/20 — a confident tetrad even
for "What is dialectical thinking?" — and echoed the question as the person's position in
6/20; with T asked for as a STANCE (`THESIS_IS_A_STANCE`, both writers) the echo went to
0/17 and the turn declined 3 of 4 informational questions, while the five open forks were
still a guess (5/5 picked an option, usually the first). The gate below
(`StanceCapture`) is for those: one question, then the card on THEIR lean
(lab notes at the end of this notebook).
So the pre-MVP runs **Path A** for the screen, and Path B is what a "your recurring
blindspots" screen would run, because it is the one with scores and memory.

**Path A** asks the tool-less `Consultant` for the view — one structured turn, no
database. **Path B** keeps the utterance as an `Input` and runs the one-shot build
(`SketchTetrad`: one reasoning call finds the thesis IN the utterance and writes the
tetrad; `IntroducePolarity` and `ExpandPolarity` score, dedup, ground, validate and
commit it).


## Decisions so far — what a spec inherits, and what it still has to nail

This notebook is a HAPPY-PATH example: scripted replies stand in for a person typing, nothing
retries, nothing is persisted beyond RAM except Path B's graph, and there is no UI. It carries
every decision made while developing the entry and the card, with the measurement behind each,
so that an app spec can be written FROM it without re-deriving them. Numbers come from the files
named in the lab notes at the end, never from this prose.

### Decided, with the reason

| decision | reason (measured where it says so) |
|---|---|
| **One entry box.** Not a "question / problem / intention" menu. | The three are not equal inputs but three distances from what the writer needs — a stance the person owns. A menu asks the person to self-diagnose, which is what they are bad at. Measured: leaning questions and complaints already carry the stance; open forks and informational input do not (lab notes). |
| **The entry invites a stance.** Copy candidates, most to least directive: the stem *"I should ___"* / *"I'm going to ___"*; *"What have you made up your mind about?"*; *"What are you sure about right now?"* | A blind spot is a property of commitment: the person who has decided is the one not looking around. The stem buys T-ownership for free; it loses the "my father refuses to stop driving" shape unless free text is also allowed. **Which copy — unmeasured; the spec decides.** |
| **Intake is a structured GATE, not a conversation.** `StanceCapture` → a stance OR one question, never prose; exclusivity enforced in code. | A free turn told to "just get the point" becomes a counsellor — prompt instructions of that kind measurably do not hold in this framework. Measured: the gate never built on a guess (0 imputed on 20); where it errs it asks one question too many. |
| **At most two questions** (`MAX_INTAKE_ASKS = 2`), then build on the best guess and say "as I heard it". | Never a dead end — the one thing the esoteric apps get right. A person stuck in intake has already lost the moment. |
| **The question is chosen for the words**, from a short repertoire (fork → which way they lean; complaint about someone else → what should happen; wish or feeling → what they will do; nothing → what to type). | What a psychologist actually asks: motivational interviewing's "what's good / less good about it" is T+/T-; CBT's "what are you telling yourself" elicits the belief as a sentence; solution-focused "what do you want to be different" is the intention. The common move: get the belief said in the first person. |
| **What tarot gets right, kept; what it gets wrong, not.** Kept: one gesture, one card, read yourself into it, always a next card. Not kept: "focus on your question" as the entry. | Tarot is projective — the card is ambiguous and the person supplies the meaning. This is the inverse: the card is specific and derived from what they said. A question has no thesis. |
| **The card leads with A+** ("what you might be missing"); the rest is context; no framework terminology ever. | The eye-opening is A+ against a T the person recognises as theirs. Built on a guessed T it hands them the constructive side of a position they do not hold — the reveal inverted. Hence everything above about owning T. |
| **T is asked for as a STANCE** in both writers (`THESIS_IS_A_STANCE`). | Measured: before it the view turn echoed the question as T in 6/20 ("Why keep paying when he never calls me" shown as the person's position); after it 0/17, and the turn declines informational input it had drawn before. |
| **Polishing by five fixed reactions** — `accept` · `another` · `deeper` · `misread` · `again` — not chat. | The promise is not "a tetrad" but the one that lands. Two kinds, kept apart: `another`/`accept` choose WHICH tension; `deeper`/`misread`/`again` change how the same material is read. The reaction vocabulary is also the instrument: which reaction dominates says which failure dominates. |
| **No "is this your position? ✓/✗" on every card.** Confirm only where the gate was not sure. | For an input that already carries a stance it is pure friction; the card shows "your position" and `misread` is the escape. |
| **Best-of-3 on every draw**, selection by the framework's own coherence check; runners-up serve `again`/`another` with no new reasoning call. | Measured on 40 utterances: coherent first tetrads 25/40 → 36/40 at ~15 s (Path A); 20/40 → 30/40 at ~34 s (Path B). Selection, never regeneration from the verdict (the judge passes mirrors more readily than positions). |
| **Path A for the screen; Path B for memory.** | Path A is better AND faster on the measured numbers and has no database, no Case lifecycle, no drain. Path B buys one thing — memory ("your recurring blindspots") — at 2× latency and a Case per person. Sequencing options: A first, B later as a write-through (double provider spend per utterance) or B replacing A (accept ~34 s). |
| **The empty state is "asked twice, still nothing."** | The writers' own refusal is not an app state: before the stance ask the view turn never declined (20/20); after it, it declines informational input, and the gate had already asked twice by then. |

### What the spec still has to nail

- **Entry copy** (stem vs open question), and whether free text is allowed beside the stem.
- **How the five reactions are elicited** (buttons, gestures, a word), where the optional free-text note goes, and whether `another` offers one or several.
- **How the gate's question is shown** — inline in the same box, as the first "card", or as a reply — and the "as I heard it" marker on a best-guess card.
- **Copy for the twice-asked empty state**, and for "nothing drawn" on Path A.
- **Latency budget and progress**: gate ~2.5 s, card ~15 s (A) / ~34 s (B); the `sid:progress` channel exists for B; A has nothing to show meanwhile.
- **Memory**: whether Path B runs at all in v1; if so, Case per person, `scope(sid)` per request, `principal="human"`, `drain_deferred_work` at shutdown; the map screen (`exploration_view()`) and the journey record (cards + reasons) a host keeps itself.
- **Instrumentation**: reaction distribution per card position, gate fire rate by shape, `misread` rate, time-to-accept — the numbers that decide the next prompt change.
- **Still unmeasured** (lab notes): the gate on a FRESH question set; the belief/complaint under-capture; A+ quality on real users' phrasing — the 40 statements and 20 questions are the author's sample.
- Identity, mobile, accounts, abuse — ordinary app concerns, none framework-specific.


In [ ]:
from dialectical_framework.dialectical_reasoning import DialecticalReasoning
from dialectical_framework.settings import Settings

# DI setup, once. Reads .env (host obligation 1 — see examples/advisor_chat.py).
container = DialecticalReasoning.setup(Settings.from_env())
print("model:", container.settings().ai_model)

In [ ]:
UTTERANCE = "I should quit my job and start my own company."


## The renderer

Both paths return `graph/views.py::PerspectiveView`. A consumer app never shows
framework terminology, so the view goes through `without_terminology()` — the same
projection the Advisor applies under `hides_terminology` — and the card names the
corners in plain words. `intent` is the reading (the two axes the tetrad was read
along); `complete` says whether all six corners exist.


In [ ]:
from IPython.display import Markdown, display

from dialectical_framework.graph.views import PerspectiveView


def render(view: PerspectiveView, title: str = "") -> None:
    """One card per perspective: A+ first, the two traps after, the poles as context."""
    v = view.without_terminology()
    text = lambda pole: pole.text if pole else "_(not drawn)_"
    lines = [f"### {title}" if title else ""]
    lines += [
        f"**What you might be missing:** {text(v.a_plus)}",
        "",
        f"- Your position: {text(v.t)}",
        f"- What it stands against: {text(v.a)}",
        f"- Your position, developed well: {text(v.t_plus)}",
        "",
        f"- Trap on your side: {text(v.t_minus)}",
        f"- Trap on the other side: {text(v.a_minus)}",
    ]
    if v.intent:
        lines += ["", f"_{v.intent}_"]
    if not v.complete:
        lines += ["", "_(unfinished tetrad — some corners were not drawn)_"]
    display(Markdown("\n".join(lines)))

## Path A — the Consultant's view turn (no graph) — the pre-MVP path

The utterance is the only message in the conversation; `exploration_view(focus=…)` is
three structured, thinking turns in parallel (copies of the same one-message history) that
build the tetrad; the framework's coherence check keeps the best and the record lands in
`messages` once. Nothing is validated and nothing is kept. ~15 s median.


In [ ]:
from dialectical_framework.agents.apps import COUNSELOR_PERSONA
from dialectical_framework.agents.consultant.consultant import Consultant

FOCUS = (
    "the perspective for what I just said: my position, what it stands against, "
    "and above all the constructive side of that opposing view that I am not seeing"
)


async def blindspot_via_consultant(text: str) -> PerspectiveView | None:
    head = Consultant(app_preamble=COUNSELOR_PERSONA, messages=[{"role": "user", "content": text}])
    view = await head.exploration_view(focus=FOCUS)
    return view.perspectives[0] if view.perspectives else None


a = await blindspot_via_consultant(UTTERANCE)
render(a, "Path A — Consultant") if a else print("nothing drawn")

## The gate — is there a position in it yet?

A person types whatever they type, and most of it is not a statement. Measured on 20
questions (`tests/e2e/probe_question_entry.py`, 2026-10-04) the view turn **never
declines**: 20/20 drew, including "What is dialectical thinking?" and "How much runway
should a seed startup keep?" — a confident tetrad around a stance nobody took — and on the
five open forks ("Zurich or Vilnius?") it picked an option, usually the first, as the
person's position. With the stance ask in the writers the turn now declines 3 of those 4
and echoes nothing, but the forks are unchanged — the words carry no lean for a writer to
read. The card is A+ against a T the person owns; built on a guessed T it hands
them the constructive side of a position they do not hold. So before either path runs, ONE
structured call (`concerns/stance_capture.py::StanceCapture`) reads what they typed and
returns **either a stance or one question — never prose**, because a free turn told to
"just get the point" becomes a counsellor. Three shapes:

| shape | what the words carry | what the person sees |
|---|---|---|
| `stance` | a position — stated, leaning through a question, presupposed by a complaint | the card, at once |
| `fork` | open between options, no lean | "Which way are you leaning, before you think it through?" |
| `none` | nothing anyone holds — information, a definition, a recipe | what to type instead, in one line |

The question is chosen for the words (a complaint about someone else → "what do you think
should happen?"; a wish → "what are you going to do about it?"). Capped at
`MAX_INTAKE_ASKS = 2`; on the last pass the build runs on the best guess and the card says
"as I heard it" — a person stuck in intake has already lost the moment. A leaning question
("Isn't it time we stopped standups?") never sees the gate's question: the stance is in the
words and the writers read it the same way the gate does (`THESIS_IS_A_STANCE` is one text
for all three). Measured on the same 20 (lab notes): forks 5/5 → one question, informational 4/4,
every stance it captured the asker's own lean (6/6, none imputed); where it errs it asks one
question too many (beliefs, complaints), never builds on a guess. 2.5 s a pass.


In [ ]:
from dialectical_framework.concerns.stance_capture import MAX_INTAKE_ASKS, StanceCapture


class Intake:
    """The person's turns so far, and the gate over them. The exchange becomes the
    conversation both paths read: Path A gets it as `messages`, Path B as the material."""

    def __init__(self, text: str) -> None:
        self.turns: list[str] = [text]
        self.asks: list[str] = []
        self.stance: str | None = None

    @property
    def messages(self) -> list[dict]:
        out: list[dict] = []
        for i, turn in enumerate(self.turns):
            out.append({"role": "user", "content": turn})
            if i < len(self.asks):
                out.append({"role": "assistant", "content": self.asks[i]})
        return out

    async def step(self) -> str | None:
        """None when a stance was found (it is in `self.stance`); else the question to show."""
        captured = await StanceCapture().resolve(self.turns)
        if captured.found or len(self.asks) >= MAX_INTAKE_ASKS:
            # Found — or out of questions. With no stance the build runs free-form
            # on the whole exchange (the writer's own best guess) and the card says so.
            self.stance = captured.stance
            return None
        self.asks.append(captured.ask)
        return captured.ask

    def answer(self, text: str) -> None:
        self.turns.append(text)


def focus_for(stance: str) -> str:
    """Path A's ask with the position pinned, so the turn develops THIS stance."""
    return (
        f"the perspective for the position I hold — «{stance}» — what it stands against, "
        "and above all the constructive side of that opposing view that I am not seeing"
    )


async def blindspot_gated(text: str, *, answers: tuple[str, ...] = ()) -> PerspectiveView | None:
    """The whole entry: gate, at most two questions, then Path A on the stance.
    `answers` stands in for the person typing back (a notebook has no input box)."""
    intake = Intake(text)
    answers = list(answers)
    while (ask := await intake.step()) is not None:
        print(f"  gate asks: {ask}")
        if not answers:
            print("  (no answer given — the next pass builds on the best guess)")
            intake.answer("")
            continue
        reply = answers.pop(0)
        print(f"  person   : {reply}")
        intake.answer(reply)
    print(f"  stance   : {intake.stance or '(none captured — building on the best guess, as I heard it)'}")
    head = Consultant(app_preamble=COUNSELOR_PERSONA, messages=intake.messages)
    view = await head.exploration_view(focus=focus_for(intake.stance) if intake.stance else FOCUS)
    return view.perspectives[0] if view.perspectives else None


# A leaning question: the stance is in the words, no question asked.
g1 = await blindspot_gated("Isn't it time we stopped doing daily standups?")
render(g1, "gated — leaning question") if g1 else print("nothing drawn")

# An open fork: one question, the person's answer, then the card on THEIR lean.
g2 = await blindspot_gated("Should I take the Zurich offer or stay in Vilnius?", answers=["Vilnius, if I'm honest — but I feel I'd regret it."])
render(g2, "gated — fork, answered") if g2 else print("nothing drawn")

# Nothing to hold: the gate says what to type instead.
g3 = await blindspot_gated("What is dialectical thinking?")
render(g3, "gated — no stance, best guess") if g3 else print("nothing drawn")


## Polishing — Path A

The promise of the app is not "a tetrad" but the one that lands: *"that's a good one,
I hadn't seen it."* So the first card is a draft and the person's reaction steers the
next. This is not a chat — the person never talks to a counsellor, they react to a card —
and the reactions are a fixed vocabulary of five, each a different recalculation:

| reaction | the person means | what is recalculated |
|---|---|---|
| `accept` | that's it | nothing — the card stands |
| `another` | not my fight / I know this one | a **different tension** from the same material |
| `deeper` | sharper, less safe | the **same tension at higher resolution**, in their own particulars |
| `misread` | that is not my position | the **position re-read** from the material, then its other side |
| `again` | say it differently | the **same tension re-worded**; the model's variance as a feature |

Two meanings, kept apart on purpose: `another`/`accept` are about WHICH tension,
`deeper`/`misread`/`again` about how the same material is read. How a UI elicits these
is the app's business; the framework sees five reasons.

On Path A the Consultant holds the conversation: `exploration_view()` records what it
drew, in words, so the next call with the reaction as `focus` draws AGAINST what was
shown. Nothing is kept beyond `head.messages`.


In [ ]:
class PolishA:
    """One Consultant, many cards. Each reaction becomes the next view's focus."""

    def __init__(self, text: str) -> None:
        self.head = Consultant(app_preamble=COUNSELOR_PERSONA, messages=[{"role": "user", "content": text}])
        self.cards: list[PerspectiveView] = []

    async def first(self) -> list[PerspectiveView]:
        return await self._draw(FOCUS)

    def accept(self) -> PerspectiveView | None:
        """That's it. Nothing recalculated; the last card stands."""
        return self.cards[-1] if self.cards else None

    async def another(self, note: str = "", how_many: int = 1) -> list[PerspectiveView]:
        """Not my fight / I know this one — a DIFFERENT tension, or several."""
        count = {1: "one tension", 2: "two different tensions", 3: "three different tensions"}.get(how_many, "up to four different tensions")
        return await self._draw(
            f"The person set aside what was shown{self._quote(note)}. Draw {count} they have NOT seen "
            "yet — a different tension in what they said, not the shown one re-worded — and above "
            "all the constructive side of the opposing view they are not seeing."
        )

    async def deeper(self, note: str = "") -> list[PerspectiveView]:
        """Sharper: the same tension at higher resolution."""
        return await self._draw(
            f"The person wants the SAME tension as the last card, sharper{self._quote(note)}: the two "
            "constructive sides and the two traps in their own particulars, less safe, more pointed. "
            "Keep the position and what it stands against; raise the resolution."
        )

    async def misread(self, note: str = "") -> list[PerspectiveView]:
        """That is not my position: re-read it from what they said, then its other side."""
        return await self._draw(
            f"The position shown is NOT what the person holds{self._quote(note)}. Re-read their position "
            "from what they said, find what it genuinely stands against, and draw that tetrad."
        )

    async def again(self) -> list[PerspectiveView]:
        """Say it differently: the same tension, fresh wording, same meaning."""
        return await self._draw(
            "Draw the SAME tension as the last card, said differently: fresh wording for every "
            "position, the same meaning, nothing added."
        )

    @staticmethod
    def _quote(note: str) -> str:
        return f' — in their words: "{note}"' if note.strip() else ""

    async def _draw(self, focus: str) -> list[PerspectiveView]:
        view = await self.head.exploration_view(focus=focus)
        start = len(self.cards)
        self.cards.extend(view.perspectives)
        for i, pv in enumerate(view.perspectives, 1):
            render(pv, f"Path A — card {start + i}")
        if not view.perspectives:
            print("nothing drawn")
        return list(view.perspectives)


polish_a = PolishA(UTTERANCE)
await polish_a.first()


In [ ]:
# Not convinced. Two different tensions, then the same one sharper.
await polish_a.another("Too obvious — I know the security argument.", how_many=2)
await polish_a.deeper()


## Path B — the one-shot build into the graph — scores and memory; not the pre-MVP path

What the Advisor's `anchor` runs since `fac4909`, called headlessly inside a Case. The
utterance is NOT the thesis: it is the person's material, kept verbatim as an `Input`
(`AddInput`), and the build's one reasoning call finds the position in it — the thesis
is distilled, not pasted. Then the graph's checks run on what came out: both poles are
classified against the systemic taxonomy, the antithesis scored (HS, Mode, Arousal), the
four aspects scored (complementarity), the tetrad grounded in the utterance's particulars,
validated (coherence + the empirical inequalities), and committed — so a second
utterance from the same person lands next to the first, traces to its own Input, and
`exploration_view()` becomes their map of recurring blindspots.

The host owns the Case (the framework never creates one) and every call runs inside
`scope(sid)`. ~34 s today (three reasoning calls in parallel plus the judge, the rest the
checks, which overlap the opposition's scoring); the `sid:progress` channel exists to show
the person what is happening meanwhile.


In [ ]:
from dialectical_framework.agents.analyst.skills.sketch_tetrad import SketchTetrad
from dialectical_framework.concerns.add_input import capture_input
from dialectical_framework.graph.nodes.case import Case
from dialectical_framework.graph.scope_context import scope
from dialectical_framework.graph.views import exploration_view, perspective_view

# One Case per person. Kept across calls: that is where the memory lives.
case = Case()
case.commit()
print("sid:", case.sid)


async def blindspot_via_pipeline(text: str) -> PerspectiveView | None:
    with scope(case.sid):
        # Whatever the person typed or pasted is MATERIAL, not the thesis: one
        # Input (added and digested), which the build reads and distils the
        # position from. The tetrad traces to it.
        source = await capture_input(text)
        # Behind the gate, pass its stance as `thesis=` — the build then words THAT
        # position in the person's terms instead of finding one in the material.
        skill = SketchTetrad(input_hashes=[source.hash])
        perspectives = await skill.resolve()
        if not perspectives:
            print("nothing built:", skill.report.summary)
            return None
        return perspective_view(perspectives[0])


b = await blindspot_via_pipeline(UTTERANCE)
render(b, "Path B — one-shot build") if b else None


## Polishing — Path B

The same five reactions, with memory. Every card is a Perspective in the Case; a card
the person moves on from is **set aside with the reaction as its reason** (the graph's
soft-mark — nothing is deleted, and the Advisor would not show it again as new), and the
next build gets the reaction as `context`, the particulars slot every prompt composes
ahead of the material. `deeper` and `again` pin the position so the tension holds;
`another` and `misread` leave it free. The record of reasons is what makes the app an
instrument for the framework: which reaction dominates says which failure dominates.

**Runners-up first.** Every build draws three and keeps one; the other two stay on
`skill.alternatives` with their coherence verdicts. `again` (say it differently) serves a
runner-up with the SAME position, `another` one with a DIFFERENT position — persisted
through `SketchTetrad(sketch=)` with every check but no new reasoning call, so the next
card is there in the time the checks take. When no runner-up fits, a fresh build runs as
before. A served runner-up is not a strawman by construction: it lost only on the
coherence read, by a margin the verdict shows.


In [ ]:
class PolishB:
    """One Case, one Input, many builds. Every reaction is a reason on the graph."""

    def __init__(self, text: str) -> None:
        self.text = text
        self.source = None
        self.cards: list = []        # (Perspective, PerspectiveView)
        self.reasons: list[str] = []  # one per card set aside, in order
        self.spare: list = []        # the last build's runners-up: (sketch, verdict)

    async def first(self):
        with scope(case.sid):
            self.source = await capture_input(self.text)
        return await self._build(context="")

    def accept(self):
        """That's it. The last card stands; nothing is built."""
        return self.cards[-1][1] if self.cards else None

    async def another(self, note: str = ""):
        """Not my fight / I know this one — a DIFFERENT tension."""
        self._set_aside("another", note)
        spare = self._take_spare(same_position=False)
        if spare is not None:
            return await self._persist(spare, f"another{self._quote(note)}")
        return await self._build(
            f"Already shown to the person: {self._shown()}. They set the last one aside{self._quote(note)}. "
            "Build a DIFFERENT tension in the material — not the shown one re-worded."
        )

    async def deeper(self, note: str = ""):
        """Sharper: the same tension at higher resolution; the position stays pinned."""
        last = self._set_aside("deeper", note)
        return await self._build(
            f"The person wants the same tension sharper{self._quote(note)}: the two constructive sides "
            "and the two traps in their own particulars, less safe, more pointed. Keep what the position "
            "stands against.",
            thesis=last.t.text if last and last.t else None,
        )

    async def misread(self, note: str = ""):
        """That is not my position: re-read it from the material, then its other side."""
        self._set_aside("misread", note)
        return await self._build(
            f"The position shown{self._shown(last_only=True)} is NOT what the person holds{self._quote(note)}. "
            "Re-read their position from the material, then find what it genuinely stands against."
        )

    async def again(self):
        """Say it differently: the same tension, fresh wording; the position stays pinned."""
        last = self._set_aside("again", "")
        spare = self._take_spare(same_position=True)
        if spare is not None:
            return await self._persist(spare, "again")
        return await self._build(
            f"The same tension as{self._shown(last_only=True)}, said differently: fresh wording for every "
            "position, the same meaning, nothing added.",
            thesis=last.t.text if last and last.t else None,
        )

    # --- the record ---

    def _set_aside(self, reaction: str, note: str):
        """Mark the last card with the reaction (and the person's words) as its reason."""
        if not self.cards:
            return None
        pp, view = self.cards[-1]
        reason = f"{reaction}: {note}".rstrip(": ")
        with scope(case.sid):
            pp.discarded = reason
            pp.save()
        self.reasons.append(reason)
        return view

    def _shown(self, last_only: bool = False) -> str:
        cards = self.cards[-1:] if last_only else self.cards
        return " " + "; ".join(f"«{pv.t.text} vs {pv.a.text}»" for _pp, pv in cards if pv.t and pv.a)

    @staticmethod
    def _quote(note: str) -> str:
        return f' — in their words: "{note}"' if note.strip() else ""

    # --- the runners-up ---

    def _take_spare(self, same_position: bool):
        """A runner-up whose position is (or is not) the last card's, removed from the spares."""
        last = self.cards[-1][1] if self.cards else None
        shown = (last.t.text if last and last.t else "").strip().lower()
        for i, (sketch, verdict) in enumerate(self.spare):
            if (sketch.thesis.strip().lower() == shown) == same_position:
                del self.spare[i]
                floor = f"{verdict.floor:.2f}" if verdict else "unjudged"
                print(f"(serving a runner-up of the last build — coherence floor {floor}, no new reasoning call)")
                return sketch
        return None

    async def _persist(self, sketch, context: str):
        """A drawn tetrad through every check and into the Case, without re-drawing it."""
        return await self._run(SketchTetrad(input_hashes=[self.source.hash], context=context, sketch=sketch))

    async def _build(self, context: str, thesis: str | None = None):
        return await self._run(SketchTetrad(input_hashes=[self.source.hash], context=context, thesis=thesis))

    async def _run(self, skill):
        with scope(case.sid):
            pps = await skill.resolve()
            if not pps:
                print("nothing built:", skill.report.summary)
                return None
            view = perspective_view(pps[0])
        if skill.alternatives:
            self.spare = list(skill.alternatives)  # a fresh build replaces the spares
        self.cards.append((pps[0], view))
        render(view, f"Path B — card {len(self.cards)}")
        return view


polish_b = PolishB(UTTERANCE)
await polish_b.first()


In [ ]:
# `another` serves a runner-up with a different position when the build left one, else builds.
await polish_b.another("Too obvious — I know the security argument.")
# `again` serves a runner-up with the same position when one is left, else builds.
await polish_b.again()
await polish_b.deeper()


### What Path B knows that Path A cannot say

The same view, before `without_terminology()`: the antithesis's HS (how well the
opposition sits against the taxonomy apex), the complementarity of each corner, the
tetrad's SP, and the validation verdict. None of this exists on Path A — its numbers
are `None` because nothing scored them, not because they are zero.


In [ ]:
if b:
    print("HS(A)        :", b.a.hs)
    print("Ks per corner:", {n: round(p.ks, 2) for n, p in b.poles.items() if p.ks is not None})
    print("metrics      :", b.metrics)
    print("validation   :", b.validation)

### The map after polishing (Path B only)

Two records of the journey. The **view** (`exploration_view()`) shows the standing
cards only — a rejected framing is set aside on the graph, with the person's reason on
the node, and the active view does not show it again as if it were new (the same rule
the Advisor's context follows). The **journey** — every card and every reaction — is
the host's own record, `polish_b.cards` and `polish_b.reasons`, which a "your
blindspots" screen would keep alongside the map.


In [ ]:
with scope(case.sid):
    the_map = exploration_view()
print(f"standing: {len(the_map.perspectives)} card(s); set aside: {sum(1 for pp, _ in polish_b.cards if pp.discarded)}")
for i, p in enumerate(the_map.perspectives, 1):
    render(p, f"standing #{i}")
for (pp, pv), reason in zip(polish_b.cards, polish_b.reasons):
    if pp.discarded:
        print(f"set aside — «{pv.t.text} vs {pv.a.text}» — because: {reason}")


## Lab notes: the entry — how the gate came to be

Lab notes for this notebook's first screen, kept HERE because the pre-MVP is an example and
its development record belongs with it. Rules in CLAUDE.md ("Shared Rendering", the view-turn
paragraph); numbers come from the files named, never from this prose.

### The question (2026-10-04, session "app-core")

The pre-MVP takes whatever the person types and asks the Consultant's view turn
for the tetrad around it. The 40 bench utterances
(`tests/e2e/probe_tetrad_quality.py` SET_A + SET_B) are 36 statements and 4
questions; on the 4, the latest best-of-3 run (`view_turn_attempts3-20261004-104227.json`)
drew a coherent tetrad with a genuine opposing position every time — by picking
the option the question NAMED ("Should I move to Berlin?" → T "Take the Berlin
job"). Whether that is the person's lean or a guess is what an intake would
exist to decide. The owner's framing: people will enter a question, a problem
or an intention, and the app should react conversationally until it has the
point, then run the algorithm. The design answer argued in the session: the
intake must be a STRUCTURED gate (a stance or one question), never a free
turn, because a free turn told to "just get the point" becomes a counsellor —
the same measured failure as every tool-election instruction. Before writing
the gate, measure what the writer does with questions.

### Instrument: `tests/e2e/probe_question_entry.py`

20 questions, pre-registered by what they CARRY before any draw — 6 *leaning*
(the wording leans: "Isn't it time we stopped standups?"), 5 *neutral forks*
("Zurich or Vilnius?"), 3 *complaints* about someone else ("Why does my manager
take credit…"), 4 *none* (informational: "What is dialectical thinking?"), 2
*beliefs* ("Is it normal to feel nothing when…"). Drawn exactly as the pre-MVP
draws (`Consultant(app_preamble=COUNSELOR_PERSONA)`, `exploration_view(focus=FOCUS)`,
best-of-3), then three reads per draw: the pipeline's coherence judge
(`_judge`), the antithesis-kind auditor (`_audit_kind`) — both the archive's,
so the figures compare with the statements — and a NEW stance auditor: how does
T relate to the question (`carried_lean` / `option_named` / `imputed` /
`not_a_stance`) and is it the asker's own position. The stance auditor is one
judge (Fable 5), unvalidated against a hand count; two of its twenty calls are
arguable ("What should I cook?" → `carried_lean`; "I want my daughter to talk
to me" → `not_a_stance`), so read its counts as ±2. Every item persisted.

### Before the stance ask — `question_entry-20261004-143700.json`

| | questions (n=20) | statements, same path, best-of-3 (n=40) |
|---|---|---|
| drawn | **20/20** — never empty | 40/40 |
| CC pass | 17/20 (85%, 95% 64–95%) | 36/40 |
| antithesis a genuine position | 18/20 (90%) | 36/40 |
| T is the asker's own stance | 17/20 | — |

Stance: `carried_lean` 7, `option_named` 5, `not_a_stance` 6, `imputed` 2.
Median 15.0 s a draw (statements: ~15 s).

| category | n | CC | T was |
|---|---|---|---|
| leaning | 6 | 4/6 | 4 carried the lean; 2 echoed the question |
| neutral fork | 5 | 5/5 | 5/5 an option the question named (4 of 5 the first one) |
| complaint | 3 | 3/3 | 1 own stance, 1 imputed diagnosis (not the asker's), 1 wish |
| none | 4 | 4/4 | drew a confident tetrad every time; 2 `not_a_stance`, 1 `imputed`, 1 `carried_lean` |
| belief | 2 | 1/2 | 1 lean, 1 echo |

Three findings.

1. **The tetrad machinery does not care that it is a question.** Coherence and
   antithesis kind are statement-grade. The draw is not the defect.
2. **"Nothing to draw yet" does not exist on the view turn.** The notebook
   claimed a bare question may draw nothing; 0/20 came back empty. "What is
   dialectical thinking?" got T "Dialectical thinking: holding opposed positions
   in structured tension" and A+ "Delivers a clear, actionable call fast"; "How
   much runway should a seed startup keep?" got T "Keep long runway to survive",
   imputed. Two of the four no-stance questions nonetheless produced cards a
   person might find useful (Spanish: method vs immersion; dinner: cook it all
   vs potluck) — which is the trap: plausible nonsense shown as "your position".
   So the gate is load-bearing for this category, not a safety net, and the
   empty state is the GATE's, not the writer's.
3. **T echoed the question in 6/20, and that is a WRITER defect, not a gate
   defect.** "Why should I keep paying for my adult son's phone when he never
   calls?" → T "Why keep paying when he never calls me". The lean ("I should
   stop paying") was in the words and the writer transcribed the question. The
   A+ was still fine ("Keep a safety net open without demanding he earn it"),
   but the person reads a question back as their own position — the `misread`
   tap the app most wants to avoid.
4. **Open forks always get a guess, usually the first option** (5/5). The
   tetrad holds both options either way, so the card is not wrong — but the
   headline is A+, and if the guessed T is the option they were NOT leaning to,
   the headline hands them the constructive side of their own lean. Eye-opening
   inverted. This is the gate's "which way are you leaning?" case, and "should I
   X or Y" is likely the most common real question form.

### The two changes

**`view_sketch.THESIS_IS_A_STANCE`** — one text, three sites: the view turn's
request (beside `_OPPOSING_POSITION_ASK`), `tetrad_sketch_prompt`'s free-thesis
branch (a GIVEN thesis is kept as given), and the gate's system prompt. It says
what T is (a claim the person could say in the first person), gives the three
question→lean examples from the run, names the fork case, and says a request
for information carries no stance. The view turn's "nothing at all rather than
a tension the conversation does not support" rule now names that case.

**`concerns/stance_capture.py::StanceCapture`** — one json-structured call over
the person's turns so far → `StanceCaptured(shape, stance, ask)`, exactly one
of `stance`/`ask` set, enforced in `_settle` (a blank stance under the `stance`
shape is NONE; a stance written under the wrong shape is not built on).
`FORK_ASK` / `NO_STANCE_ASK` are the gate's own questions when the model names
the shape and leaves `ask` empty; the prompt carries a menu of which question
fits which words (fork → the lean; complaint → what should happen; wish → what
they will do; nothing → what to type). `MAX_INTAKE_ASKS = 2` is a module
constant: on the last pass the host builds on the best guess and the card says
"as I heard it". The notebook's `Intake` is the reference loop; the stance is
pinned as `focus` on Path A and as `thesis=` on Path B. Nothing persisted.

### After the stance ask — `question_entry-20261004-145135.json`

Same 20, same instruments, fresh judge, 4 min 27 s, median 15.2 s a draw.

| | before | after |
|---|---|---|
| drawn | 20/20 | **17/20** — the view turn declined 3 of the 4 *none* questions (Spanish, dinner, "what is dialectical thinking"); runway still drew, imputed |
| T echoed the question (`not_a_stance`) | 6/20 | **0/17** |
| T is the asker's own stance | 17/20 | 16/17 |
| `carried_lean` on the 6 leaning | 4/6 | **6/6** |
| CC pass | 17/20 (85%) | 14/17 (82%) — paired 2 up / 3 down, noise at this n |
| antithesis a genuine position | 18/20 | 15/17 (1 mirror, 1 not-an-opposition) |
| forks | 5/5 `option_named` | 5/5 `option_named` — unchanged, as expected: the ask cannot know a lean the words do not carry |

What moved is what the ask targeted: every T is now a first-person claim ("I
should stop paying for his phone", "We should stop daily standups", "I deserve
credit for the work I do"), and the view turn now uses its own "nothing at all"
rule on informational turns — so on Path A the gate's `none` branch is mostly a
second net and the FORK branch is the load-bearing one (5/5 still guessed).
Coherence and antithesis kind did not move in either direction a 20-item run
can resolve.

**One retraction from the run.** The constant as measured carried a third
example, `"Why do I keep hiring people like me?" carries "I should stop hiring
people like me"`. On that very utterance the writer then produced T "I should
stop hiring people like me" vs A "I should hire people who complement me" —
the kind auditor's one `not_an_opposition`: the example had named the
reflective side as the stance, and T and A collapsed onto it. The example was
removed after the run (the sentence "a behaviour described is not yet a
stance" stays); the shipped text differs from the measured one by that
deletion, on one item's evidence, and the next run of the probe is on the
shipped text. The before/after figures above are for the text WITH the example.

### The gate on the same 20 — `stance_gate-20261004-150956.json`

`tests/e2e/probe_stance_gate.py`: `StanceCapture` on each question (first pass), the stance
auditor on every stance it captured, and the LOOP — every fork and every informational
question gets a pre-registered scripted reply as the person's answer and the second pass
must return a stance that matches it (one small judge call). 65 s, median **2.5 s** a pass.

| | |
|---|---|
| shape agrees with the pre-registered category | 15/20 |
| forks → `fork` | **5/5** |
| informational → `none` | **4/4** |
| leaning → `stance` | 5/6 |
| complaint → `stance` | 1/3 (asked instead) |
| belief → `stance` | 0/2 (asked instead) |
| captured stances read as the lean the words carried | **6/6**, all first-person, 0 imputed |
| asked a question | 14/20 |
| second pass after the scripted reply: stance found | 8/9; matches the reply 7/8 (the one "no" — "I want to go back to university at forty" against the reply "I want to go back" — is a judge miss on inspection) |

Reading. Where the gate is wrong it is wrong in ONE direction: it asks a question where the
words arguably carried a stance (both beliefs, two of three complaints, one reflective
"why do I keep…"), never the reverse — no stance was imputed. That is the safe side: an
extra question costs 2.5 s and a tap; a false stance costs the card. The questions it
writes are specific to the words, not generic ("Are you leaning toward telling her, or
toward staying silent?"; "What dish or menu are you leaning toward?"), and three of the
four informational questions became a usable first-person stance after one reply
("I'm going to move to Madrid for the summer and learn Spanish by just speaking it") —
the never-a-dead-end property, working. The fourth ("I just wanted to know what it is")
was asked again; the host's cap then builds free-form, and since the stance ask the view
turn declines there — so the app's true empty state is "asked twice, still nothing", and
the screen needs a line for it. The belief/complaint asks could be tightened with two
examples in the gate prompt; not done, because a prompt tuned on these 20 and measured on
these 20 is overfitting, and the error direction is the tolerable one. Re-measure on a
fresh set before touching it.

### What is still not measured

- The gate on a FRESH question set (the 20 above are both the writer's and the gate's
  development sample now); the belief/complaint under-capture against it.
- Real users' phrasing. Twenty authored questions are a sample of the author.
- The stance auditor against a hand count (the kind auditor has one; this one
  does not).


## Where this goes

- **Re-measure before trusting it.** `tests/e2e/probe_view_turn_ask.py` draws the 40
  utterances the way Path A does and audits them; `tests/e2e/probe_tetrad_quality.py`
  runs Path B's build on the same 40 (`TETRAD_PROBE_ATTEMPTS=N` and `VIEW_TURN_ATTEMPTS=N`
  set the draws). Every item is kept as JSON; the best-of-3 figures are one pre-registered
  generation each (`docs/dev-notes/antithesis-selection.md`, "Best-of-N"), not a result.
- **Backend**: a small HTTP service hosting the chosen path — `POST /blindspot {text}` →
  `PerspectiveView.to_dict()`. Host obligations are the documented ones (Case per
  person, `scope(sid)` per request, `principal="human"`, drain at shutdown).
- **"Nothing to draw yet" does not happen by itself.** Measured 2026-10-04: on 20
  questions Path A drew 20/20, a confident tetrad even where nothing was a stance. The
  empty state is the GATE's (`StanceCapture`, above), not the writer's; the app's
  graceful state is the gate's one question, and the build on the best guess after it.
- **Unmeasured**: A+ quality on either path against real users' phrasing. The probe
  is the first sample, not a result.
